# 02_embeddings: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/02_embeddings/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Project real catalog embeddings with a small PCA calculation.

Concepts: centering, SVD projection, lost information.
Run: uv run python modules/02_embeddings/examples/01_from_scratch.py
Expected: 500 x 2 coordinates; PCA distance agreement
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

import numpy as np
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits

from common.settings import cache_dir
from common.teaching import catalog, evidence


def main() -> None:
    products, vectors, _, _ = catalog(500)
    centered = vectors - vectors.mean(axis=0)
    with threadpool_limits(limits=1):
        # KEY IDEA: the first two singular directions explain the most variance, not necessarily relevance.
        _, singular, directions = np.linalg.svd(centered, full_matrices=False)
        projected = centered @ directions[:2].T
        library = PCA(n_components=2, svd_solver="full").fit_transform(vectors)
    ours = np.linalg.norm(projected[1:] - projected[0], axis=1)
    theirs = np.linalg.norm(library[1:] - library[0], axis=1)
    np.testing.assert_allclose(ours, theirs, atol=1e-4)
    retained = float((singular[:2] ** 2).sum() / (singular**2).sum())
    evidence(
        "catalog_map",
        {
            "rows": len(products),
            "original_dimensions": vectors.shape[1],
            "projected_dimensions": 2,
            "variance_fraction": retained,
        },
    )
    print(projected.shape, "variance retained:", round(retained, 4))
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
    for category in sorted(products["category"].unique()):
        mask = products["category"] == category
        ax.scatter(projected[mask, 0], projected[mask, 1], s=12, alpha=0.6, label=category)
    ax.set(
        xlabel="Principal component 1",
        ylabel="Principal component 2",
        title=f"Marlowe catalog: 2 of 384 dimensions; {retained:.1%} variance retained",
    )
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
    path = cache_dir() / "tier1" / "catalog_map.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(path)
    print("Open the Playground Catalog map tab; search still uses all 384 coordinates.")


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/02_embeddings/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Reuse cached real vectors in an embedded search engine.

    Concepts: cache key, model identity, explicit vectors, model-free retrieval.
    Run: uv run --extra embedded python modules/02_embeddings/examples/02_production.py
    Expected: repeated cached arrays agree; Chroma returns mug warmers
    CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
    Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
    """

    # course-extra: embedded
    import tempfile
    import time
    import uuid
    from pathlib import Path

    import chromadb
    import numpy as np

    from common.data import product_texts
    from common.embeddings import EmbeddingCache, EmbeddingSpec
    from common.pretty import show_hits
    from common.teaching import catalog, ranked_hits

    def main() -> None:
        products, vectors, query, metadata = catalog(200)
        texts = product_texts(products)
        # These are real precomputed vectors; the provider is a lookup, not model inference.
        lookup = dict(zip(texts, vectors, strict=True))
        calls = []

        def provider(missing, role):
            calls.append(len(missing))
            return np.stack([lookup[text] for text in missing])

        with tempfile.TemporaryDirectory() as directory:
            cache = EmbeddingCache(Path(directory))
            spec = EmbeddingSpec(**metadata["model"])
            first = cache.encode(texts, spec, "document", provider)
            second = cache.encode(texts, spec, "document", provider)
            np.testing.assert_array_equal(first, second)
            assert len(calls) == 1
        client = chromadb.EphemeralClient(settings=chromadb.Settings(anonymized_telemetry=False))
        name = "cached_" + uuid.uuid4().hex
        collection = client.create_collection(
            name, embedding_function=None, configuration={"hnsw": {"space": "cosine"}}
        )
        try:
            collection.add(ids=products["id"].tolist(), embeddings=second.tolist())
            start = time.perf_counter()
            found = collection.query(query_embeddings=[query.tolist()], n_results=5)
            elapsed = (time.perf_counter() - start) * 1000
            ranked = list(zip(found["ids"][0], [1 - d for d in found["distances"][0]], strict=True))
            show_hits(ranked_hits(products, ranked), elapsed, title="Reuse embeddings")
            print("Provider calls:", len(calls))
        finally:
            client.delete_collection(name)

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/02_embeddings/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure batching and cache behavior without hidden downloads.

Concepts: cold/warm cache, role and prefix identity, optional real CPU inference.
Run: uv run python modules/02_embeddings/examples/03_measure.py
Expected: matching arrays, distinct role/prefix keys, measured cache timings
CPU: about 2–10; --live about 30–60 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

import argparse
import tempfile
import time
from dataclasses import replace
from pathlib import Path

import numpy as np

from common.bench.resources import MemorySampler
from common.embeddings import EmbeddingCache, HashEncoder, TextEncoder
from common.teaching import catalog, evidence


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--live", action="store_true", help="Use cached MiniLM; requires --extra models"
    )
    args = parser.parse_args()
    with MemorySampler() as memory:
        measurements = experiment(args.live)
    measurements["memory"] = memory.report()
    evidence("embedding_cache_live" if args.live else "embedding_cache", measurements)


def experiment(live: bool) -> dict:
    texts = catalog(64)[0]["description"].tolist()
    encoder = TextEncoder(batch_size=16) if live else HashEncoder()
    start = time.perf_counter()
    encoder.encode(texts[:1])
    startup = time.perf_counter() - start
    batches = {}
    for size in [1, 16, 32]:
        if live:
            encoder.batch_size = size
        repeats = []
        for _ in range(5):
            start = time.perf_counter()
            values = np.concatenate(
                [encoder.encode(texts[i : i + size]) for i in range(0, len(texts), size)]
            )
            repeats.append(time.perf_counter() - start)
        batches[str(size)] = repeats
        assert values.shape == (64, encoder.spec.dimensions)
    with tempfile.TemporaryDirectory() as directory:
        cache = EmbeddingCache(Path(directory))
        timings = []
        for _ in range(6):
            start = time.perf_counter()
            cached = cache.encode(texts, encoder.spec, "document", encoder.encode)
            timings.append(time.perf_counter() - start)
        np.testing.assert_allclose(values, cached, atol=2e-6)
        # KEY IDEA: adding a prefix changes the space even if dimension count stays identical.
        changed = replace(encoder.spec, query_prefix="query: ")
        assert cache.key(texts[0], changed, "query") != cache.key(texts[0], encoder.spec, "query")
        assert cache.key(texts[0], encoder.spec, "document") != cache.key(
            texts[0], encoder.spec, "query"
        )
    label = "real cached MiniLM" if live else "hash mechanics only; no semantic-quality claim"
    print(label, "batch seconds:", batches, "miss/hit seconds:", timings)
    return {
        "encoder": label,
        "startup_seconds": startup,
        "batch_seconds": batches,
        "cache_miss_hit_seconds": timings,
        "space": encoder.spec.space,
    }


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from dataclasses import replace

from common.embeddings import EmbeddingSpec

spec = EmbeddingSpec()
print(spec.space)
print(replace(spec, query_prefix="query: ").space)

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.